In [12]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. First, connect to the TiDB server WITHOUT specifying a database
# Notice we omitted '/tidb_ecommerce' at the end of the URI
BASE_URI = 'mysql+pymysql://root:@127.0.0.1:4000/'
base_engine = create_engine(BASE_URI)

# Create the database if it doesn't exist
with base_engine.begin() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS tidb_ecommerce;"))
    
print("Database 'tidb_ecommerce' created (or already exists).")

# 2. Now, connect TO the newly created database
DATABASE_URI = 'mysql+pymysql://root:@127.0.0.1:4000/tidb_ecommerce'
engine = create_engine(DATABASE_URI)

# 3. Define the schema creation statements
create_tables_sql = """
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS inventory;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS users;

CREATE TABLE users (
    user_id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(255),
    email VARCHAR(255) UNIQUE,
    region_code VARCHAR(10)
);

CREATE TABLE products (
    product_id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(255),
    category VARCHAR(100),
    price DECIMAL(10, 2)
);

CREATE TABLE inventory (
    location_id INT,
    product_id INT,
    region_code VARCHAR(10),
    quantity INT,
    PRIMARY KEY (location_id, product_id),
    CONSTRAINT fk_inventory_product 
        FOREIGN KEY (product_id) REFERENCES products(product_id) 
        ON DELETE CASCADE
);

CREATE TABLE orders (
    order_id INT AUTO_INCREMENT,
    user_id INT,
    order_time DATETIME DEFAULT CURRENT_TIMESTAMP,
    region_code VARCHAR(10),
    status VARCHAR(50),
    PRIMARY KEY (region_code, order_time, order_id),
    UNIQUE (order_id), 
    KEY idx_user_time (user_id, order_time),
    CONSTRAINT fk_orders_user 
        FOREIGN KEY (user_id) REFERENCES users(user_id) 
        ON DELETE RESTRICT
);

CREATE TABLE order_items (
    order_id INT,
    product_id INT,
    qty INT,
    unit_price DECIMAL(10, 2),
    PRIMARY KEY (order_id, product_id),
    CONSTRAINT fk_order_items_order 
        FOREIGN KEY (order_id) REFERENCES orders(order_id) 
        ON DELETE CASCADE,
    CONSTRAINT fk_order_items_product 
        FOREIGN KEY (product_id) REFERENCES products(product_id) 
        ON DELETE RESTRICT
);
"""

# 4. Execute the table creation statements
with engine.begin() as conn:
    for statement in create_tables_sql.split(';'):
        if statement.strip():
            conn.execute(text(statement))
            
print("TiDB schema successfully created via SQLAlchemy!")

# 5. Verify the tables were created successfully
verify_query = """
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'tidb_ecommerce';
"""
df_tables = pd.read_sql(verify_query, con=engine)
display(df_tables)

Database 'tidb_ecommerce' created (or already exists).
TiDB schema successfully created via SQLAlchemy!


,table_name
0,users
1,products
2,orders
3,order_items
4,inventory


In [13]:
import pandas as pd
from sqlalchemy import create_engine
import random
import string

# Establish SQLAlchemy Engine
# Note: Ensure the base database 'tidb_ecommerce' was created successfully in the previous step
DATABASE_URI = 'mysql+pymysql://root:@127.0.0.1:4000/tidb_ecommerce'
engine = create_engine(DATABASE_URI)

# Configuration for data generation
NUM_USERS = 1000
NUM_PRODUCTS = 500
REGIONS = ['US-CA', 'US-NY', 'EU-UK', 'AS-JP', 'AS-IN']

# Data dictionaries to simulate realistic data without Faker
FIRST_NAMES = ['James', 'Mary', 'John', 'Patricia', 'Robert', 'Jennifer', 'Michael', 'Linda', 'William', 'Elizabeth', 'David', 'Barbara', 'Richard', 'Susan', 'Joseph', 'Jessica', 'Thomas', 'Sarah', 'Charles', 'Karen']
LAST_NAMES = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis', 'Rodriguez', 'Martinez', 'Hernandez', 'Lopez', 'Gonzalez', 'Wilson', 'Anderson', 'Thomas', 'Taylor', 'Moore', 'Jackson', 'Martin']
DOMAINS = ['gmail.com', 'yahoo.com', 'outlook.com', 'hotmail.com', 'example.com']

ADJECTIVES = ['Wireless', 'Smart', 'Portable', 'Ergonomic', 'Compact', 'Heavy-Duty', 'Luxury', 'Digital', 'Mechanical', 'Vintage']
NOUNS = ['Headphones', 'Keyboard', 'Monitor', 'Speaker', 'Mouse', 'Chair', 'Desk', 'Camera', 'Tablet', 'Laptop']
CATEGORIES = ['Electronics', 'Clothing', 'Home', 'Books', 'Toys']

print("Generating synthetic data using standard Python libraries...")

# 1. Generate Users
users_data = []
for i in range(NUM_USERS):
    first = random.choice(FIRST_NAMES)
    last = random.choice(LAST_NAMES)
    # Adding a random number or index to ensure email uniqueness
    email = f"{first.lower()}.{last.lower()}{i}@{random.choice(DOMAINS)}"
    
    users_data.append({
        'name': f"{first} {last}",
        'email': email,
        'region_code': random.choice(REGIONS)
    })
df_users = pd.DataFrame(users_data)

# 2. Generate Products
products_data = []
for _ in range(NUM_PRODUCTS):
    prod_name = f"{random.choice(ADJECTIVES)} {random.choice(NOUNS)}"
    products_data.append({
        'name': prod_name,
        'category': random.choice(CATEGORIES),
        'price': round(random.uniform(5.0, 500.0), 2)
    })
df_products = pd.DataFrame(products_data)

# 3. Generate Inventory
# We assume product_ids will increment from 1 to NUM_PRODUCTS
locations = {r: i+1 for i, r in enumerate(REGIONS)}
inventory_data = []

for prod_id in range(1, NUM_PRODUCTS + 1):
    for region in REGIONS:
        # Not every product is in every region (80% chance to have stock)
        if random.random() > 0.2: 
            inventory_data.append({
                'location_id': locations[region],
                'product_id': prod_id,
                'region_code': region,
                'quantity': random.randint(10, 1000)
            })
df_inventory = pd.DataFrame(inventory_data)

# 4. Bulk Insert Data into TiDB using Pandas to_sql()
print("Inserting data into TiDB...")

# Write to database
df_users.to_sql('users', con=engine, if_exists='append', index=False)
print(f"Inserted {len(df_users)} users.")

df_products.to_sql('products', con=engine, if_exists='append', index=False)
print(f"Inserted {len(df_products)} products.")

df_inventory.to_sql('inventory', con=engine, if_exists='append', index=False)
print(f"Inserted {len(df_inventory)} inventory records.")

print("Database successfully populated!")

Generating synthetic data using standard Python libraries...
Inserting data into TiDB...
Inserted 1000 users.
Inserted 500 products.
Inserted 2001 inventory records.
Database successfully populated!


In [14]:
import pandas as pd

# Assuming your engine is already created from the previous steps
# engine = create_engine('mysql+pymysql://root:@127.0.0.1:4000/tidb_ecommerce')

print("--- Row Counts ---")
counts_query = """
SELECT 'users' AS table_name, COUNT(*) AS total_rows FROM users
UNION ALL
SELECT 'products', COUNT(*) FROM products
UNION ALL
SELECT 'inventory', COUNT(*) FROM inventory;
"""
df_counts = pd.read_sql(counts_query, con=engine)
display(df_counts)
print("\n--- Users Table (First 5 Rows) ---")
display(pd.read_sql("SELECT * FROM users LIMIT 5;", con=engine))

print("\n--- Products Table (First 5 Rows) ---")
display(pd.read_sql("SELECT * FROM products LIMIT 5;", con=engine))

print("\n--- Inventory Table (First 5 Rows) ---")
display(pd.read_sql("SELECT * FROM inventory LIMIT 5;", con=engine))

print("\n--- Inventory Distribution by Region ---")
# This query checks how many items and total quantity are stored in each region
region_distribution_query = """
SELECT 
    region_code, 
    COUNT(DISTINCT product_id) as unique_products_in_stock, 
    SUM(quantity) as total_items_in_warehouse
FROM inventory 
GROUP BY region_code
ORDER BY region_code;
"""
df_regions = pd.read_sql(region_distribution_query, con=engine)
display(df_regions)

print("\n--- Product Distribution by Category ---")
category_distribution_query = """
SELECT category, COUNT(*) as product_count, ROUND(AVG(price), 2) as avg_price
FROM products
GROUP BY category
ORDER BY product_count DESC;
"""
df_categories = pd.read_sql(category_distribution_query, con=engine)
display(df_categories)

--- Row Counts ---


,table_name,total_rows
0,products,500
1,users,1000
2,inventory,2001



--- Users Table (First 5 Rows) ---


,user_id,name,email,region_code
0,1,Sarah Anderson,sarah.anderson0@yahoo.com,AS-IN
1,2,Karen Johnson,karen.johnson1@gmail.com,US-CA
2,3,Sarah Gonzalez,sarah.gonzalez2@hotmail.com,US-CA
3,4,Elizabeth Taylor,elizabeth.taylor3@gmail.com,US-NY
4,5,Thomas Jackson,thomas.jackson4@hotmail.com,AS-IN



--- Products Table (First 5 Rows) ---


...

,location_id,product_id,region_code,quantity
0,1,1,US-CA,550
1,1,2,US-CA,59
2,1,3,US-CA,672
3,1,5,US-CA,127
4,1,7,US-CA,331



--- Inventory Distribution by Region ---


,region_code,unique_products_in_stock,total_items_in_warehouse
0,AS-IN,391,206651.0
1,AS-JP,378,188954.0
2,EU-UK,400,196993.0
3,US-CA,416,201689.0
4,US-NY,416,203844.0



--- Product Distribution by Category ---


,category,product_count,avg_price
0,Home,113,243.58
1,Electronics,111,254.33
2,Books,99,255.99
3,Toys,92,258.21
4,Clothing,85,266.60


In [15]:
import pandas as pd
from sqlalchemy import create_engine
import random
from datetime import datetime, timedelta

# Establish SQLAlchemy Engine
DATABASE_URI = 'mysql+pymysql://root:@127.0.0.1:4000/tidb_ecommerce'
engine = create_engine(DATABASE_URI)

print("Fetching metadata to ensure referential integrity...")

# 1. Fetch valid users and products from TiDB to satisfy Foreign Key constraints
with engine.connect() as conn:
    df_existing_users = pd.read_sql("SELECT user_id, region_code FROM users", con=conn)
    df_existing_products = pd.read_sql("SELECT product_id, price FROM products", con=conn)

valid_users = df_existing_users.to_dict('records')
# Create a dictionary to easily map product_id -> actual price
valid_products = {row['product_id']: row['price'] for _, row in df_existing_products.iterrows()}
valid_product_ids = list(valid_products.keys())

NUM_ORDERS = 3000
STATUSES = ['PENDING', 'PROCESSING', 'SHIPPED', 'DELIVERED', 'CANCELLED']

print("Generating synthetic orders and order items...")

orders_data = []
order_items_data = []

end_date = datetime.now()
start_date = end_date - timedelta(days=30)
total_seconds = int((end_date - start_date).total_seconds())

# 2. Generate Orders
for _ in range(NUM_ORDERS):
    # Pick a real user from the database
    user = random.choice(valid_users)
    user_id = user['user_id']
    
    # Ideally, orders happen in the user's region, though not strictly required
    region_code = user['region_code'] 
    
    order_time = start_date + timedelta(seconds=random.randint(0, total_seconds))
    status = random.choices(STATUSES, weights=[10, 10, 20, 50, 10])[0] 
    
    # We do NOT specify 'order_id' here. 
    # We let TiDB's AUTO_INCREMENT generate it to prevent Primary Key collisions.
    orders_data.append({
        'user_id': user_id,
        'order_time': order_time,
        'region_code': region_code,
        'status': status
    })

df_orders = pd.DataFrame(orders_data)

print("Inserting Orders into TiDB (to generate Auto-Increment IDs)...")
# Insert orders first so TiDB generates the Primary Keys
df_orders.to_sql('orders', con=engine, if_exists='append', index=False)

# Fetch the newly generated order_ids back from the database
with engine.connect() as conn:
    # Fetching the latest 3000 orders assuming we just inserted them
    df_new_orders = pd.read_sql(f"SELECT order_id FROM orders ORDER BY order_id DESC LIMIT {NUM_ORDERS}", con=conn)

new_order_ids = df_new_orders['order_id'].tolist()

# 3. Generate Order Items
for order_id in new_order_ids:
    num_items_in_order = random.randint(1, 5)
    
    # Sample real product IDs without replacement (prevents composite PK violation)
    product_ids = random.sample(valid_product_ids, num_items_in_order)
    
    for prod_id in product_ids:
        # Fetch the actual price from our products dictionary
        actual_price = valid_products[prod_id]
        
        order_items_data.append({
            'order_id': order_id,
            'product_id': prod_id,
            'qty': random.randint(1, 4),
            'unit_price': actual_price  # Enforces logical data integrity!
        })

df_order_items = pd.DataFrame(order_items_data)

print("Inserting Order Items into TiDB...")
df_order_items.to_sql('order_items', con=engine, if_exists='append', index=False)

print(f"Successfully inserted {len(df_orders)} orders and {len(df_order_items)} order items.")

Fetching metadata to ensure referential integrity...
Generating synthetic orders and order items...
Inserting Orders into TiDB (to generate Auto-Increment IDs)...
Inserting Order Items into TiDB...
Successfully inserted 3000 orders and 9057 order items.


In [16]:
print("--- Sample Joined Order History ---")
verify_join_query = """
    SELECT 
        o.order_id, 
        o.user_id, 
        o.order_time, 
        o.region_code,
        COUNT(oi.product_id) as total_unique_items,
        SUM(oi.qty) as total_items_purchased,
        SUM(oi.qty * oi.unit_price) as total_order_value
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.order_id, o.user_id, o.order_time, o.region_code
    ORDER BY total_order_value DESC
    LIMIT 10;
"""

df_joined_history = pd.read_sql(verify_join_query, con=engine)
display(df_joined_history)

--- Sample Joined Order History ---


,order_id,user_id,order_time,region_code,total_unique_items,total_items_purchased,total_order_value
0,614,511,2026-03-30 21:18:00,US-NY,5,17.0,7214.53
1,871,320,2026-03-23 17:31:30,AS-IN,4,16.0,7118.84
2,2542,183,2026-04-06 23:17:07,EU-UK,5,17.0,6683.19
3,648,860,2026-04-18 07:25:30,US-CA,5,18.0,6399.48
4,2650,787,2026-04-14 13:10:00,AS-JP,5,16.0,6054.69
5,1460,16,2026-03-31 21:47:25,US-NY,5,17.0,5907.55
6,309,6,2026-03-29 00:21:14,AS-JP,5,15.0,5883.42
7,1306,742,2026-04-06 04:44:42,AS-JP,5,14.0,5860.48
8,1730,32,2026-03-30 00:08:41,US-CA,5,18.0,5840.30
9,1333,322,2026-04-08 10:34:13,AS-JP,5,14.0,5818.60


In [17]:
from sqlalchemy import text

def place_order(engine, user_id, region_code, items):
    """
    items should be a list of dictionaries, e.g.:
    [{'product_id': 1, 'qty': 2, 'unit_price': 15.00, 'location_id': 1}]
    """
    # Using engine.begin() automatically handles the commit/rollback for the two-phase commit
    with engine.begin() as conn:
        # 1. Insert into orders
        order_insert_stmt = text("""
            INSERT INTO orders (user_id, region_code, status)
            VALUES (:user_id, :region_code, 'PENDING')
        """)
        result = conn.execute(order_insert_stmt, {"user_id": user_id, "region_code": region_code})
        order_id = result.lastrowid

        # 2. Insert order_items and update inventory for each item
        for item in items:
            oi_stmt = text("""
                INSERT INTO order_items (order_id, product_id, qty, unit_price)
                VALUES (:order_id, :product_id, :qty, :unit_price)
            """)
            conn.execute(oi_stmt, {
                "order_id": order_id,
                "product_id": item['product_id'],
                "qty": item['qty'],
                "unit_price": item['unit_price']
            })

            inv_stmt = text("""
                UPDATE inventory
                SET quantity = quantity - :qty
                WHERE location_id = :location_id AND product_id = :product_id
            """)
            conn.execute(inv_stmt, {
                "qty": item['qty'],
                "location_id": item['location_id'],
                "product_id": item['product_id']
            })
            
    print(f"Order {order_id} placed successfully. 2PC transaction committed.")
    return order_id

In [18]:
import pandas as pd

def get_user_order_history(engine, user_id):
    query = text("""
        SELECT 
            o.order_id, o.order_time, o.region_code, o.status,
            SUM(oi.qty * oi.unit_price) AS total_amount
        FROM orders AS o
        JOIN order_items AS oi ON o.order_id = oi.order_id
        WHERE o.user_id = :user_id AND o.order_time >= NOW() - INTERVAL 30 DAY
        GROUP BY o.order_id, o.order_time, o.region_code, o.status
        ORDER BY o.order_time DESC
        LIMIT 20;
    """)
    # Using pandas to execute and format the results directly
    with engine.connect() as conn:
        df = pd.read_sql(query, con=conn, params={"user_id": user_id})
    return df

In [19]:
def check_regional_inventory(engine, region_code, quantity_threshold=50):
    query = text("""
        SELECT 
            p.product_id, p.name, p.category, i.quantity, i.location_id
        FROM inventory i
        JOIN products p ON i.product_id = p.product_id
        WHERE i.region_code = :region_code AND i.quantity < :threshold;
    """)
    
    with engine.connect() as conn:
        df = pd.read_sql(query, con=conn, params={"region_code": region_code, "threshold": quantity_threshold})
    return df

In [20]:
def get_sales_summary(engine):
    query = text("""
        SELECT 
            DATE(o.order_time) AS sale_date, 
            o.region_code,
            SUM(oi.qty * oi.unit_price) AS daily_revenue
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        GROUP BY DATE(o.order_time), o.region_code
        ORDER BY sale_date DESC, daily_revenue DESC;
    """)
    
    with engine.connect() as conn:
        df = pd.read_sql(query, con=conn)
    return df

In [21]:
# 1. Place a mock order
mock_items = [
    {'product_id': 12, 'qty': 2, 'unit_price': 15.99, 'location_id': 1},
    {'product_id': 45, 'qty': 1, 'unit_price': 299.99, 'location_id': 1}
]
place_order(engine, user_id=10, region_code='US-CA', items=mock_items)

# 2. View Order History
display(get_user_order_history(engine, user_id=10))

# 3. View Regional Inventory
display(check_regional_inventory(engine, region_code='US-CA', quantity_threshold=20))

# 4. View Global Sales Summary
display(get_sales_summary(engine))

Order 3001 placed successfully. 2PC transaction committed.


,order_id,order_time,region_code,status,total_amount
0,3001,2026-04-20 12:41:09,US-CA,PENDING,331.97
1,2379,2026-04-19 22:35:45,EU-UK,DELIVERED,183.49
2,382,2026-04-19 18:34:59,EU-UK,DELIVERED,3573.12
3,634,2026-04-09 20:00:02,EU-UK,DELIVERED,1566.08
4,1158,2026-03-29 22:18:54,EU-UK,DELIVERED,2807.04


,product_id,name,category,quantity,location_id
0,119,Portable Mouse,Electronics,12,1
1,202,Smart Tablet,Home,12,1
2,337,Heavy-Duty Mouse,Home,14,1
3,344,Wireless Camera,Home,13,1


,sale_date,region_code,daily_revenue
0,2026-04-20,AS-IN,22641.64
1,2026-04-20,AS-JP,17261.59
2,2026-04-20,EU-UK,15544.35
3,2026-04-20,US-NY,9634.40
4,2026-04-20,US-CA,8115.95
...,...,...,...
150,2026-03-21,EU-UK,20415.98
151,2026-03-21,US-CA,19861.05
152,2026-03-21,AS-IN,11676.28
153,2026-03-21,US-NY,11575.66


In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import pandas as pd
from sqlalchemy import create_engine, text
import re

# --- 1. Core Database & Validation Functions ---

def get_all_products(engine):
    query = text("SELECT product_id, name, price FROM products")
    with engine.connect() as conn:
        return pd.read_sql(query, con=conn)

def user_exists(engine, user_id):
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1 FROM users WHERE user_id = :uid"), {"uid": user_id}).scalar()
        return bool(result)

def email_exists(engine, email):
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1 FROM users WHERE email = :email"), {"email": email}).scalar()
        return bool(result)

def check_stock(engine, product_id, location_id, requested_qty):
    with engine.connect() as conn:
        stock = conn.execute(
            text("SELECT quantity FROM inventory WHERE product_id = :pid AND location_id = :lid"),
            {"pid": product_id, "lid": location_id}
        ).scalar()
        if stock is None: return False, f"Location {location_id} does not stock this product."
        if stock < requested_qty: return False, f"Insufficient stock! Only {stock} available at Location {location_id}."
        return True, "OK"

def add_user(engine, name, email, region_code):
    # engine.begin() automatically issues a COMMIT to TiDB upon successful completion
    with engine.begin() as conn:
        stmt = text("INSERT INTO users (name, email, region_code) VALUES (:name, :email, :region_code)")
        result = conn.execute(stmt, {"name": name, "email": email, "region_code": region_code})
        return result.lastrowid

def add_product(engine, name, category, price):
    # engine.begin() automatically issues a COMMIT to TiDB upon successful completion
    with engine.begin() as conn:
        stmt = text("INSERT INTO products (name, category, price) VALUES (:name, :category, :price)")
        result = conn.execute(stmt, {"name": name, "category": category, "price": price})
        return result.lastrowid

def add_inventory(engine, product_id, location_id, region_code, quantity):
    """UPSERT inventory into TiDB"""
    with engine.begin() as conn:
        stmt = text("""
            INSERT INTO inventory (product_id, location_id, region_code, quantity) 
            VALUES (:product_id, :location_id, :region_code, :quantity)
            ON DUPLICATE KEY UPDATE quantity = quantity + :quantity
        """)
        conn.execute(stmt, {
            "product_id": product_id, 
            "location_id": location_id, 
            "region_code": region_code,
            "quantity": quantity
        })

def place_order(engine, user_id, region_code, items):
    with engine.begin() as conn:
        order_insert_stmt = text("INSERT INTO orders (user_id, region_code, status) VALUES (:user_id, :region_code, 'PENDING')")
        result = conn.execute(order_insert_stmt, {"user_id": user_id, "region_code": region_code})
        order_id = result.lastrowid

        for item in items:
            oi_stmt = text("INSERT INTO order_items (order_id, product_id, qty, unit_price) VALUES (:order_id, :product_id, :qty, :unit_price)")
            conn.execute(oi_stmt, {"order_id": order_id, "product_id": item['product_id'], "qty": item['qty'], "unit_price": item['unit_price']})

            inv_stmt = text("UPDATE inventory SET quantity = quantity - :qty WHERE location_id = :location_id AND product_id = :product_id")
            conn.execute(inv_stmt, {"qty": item['qty'], "location_id": item['location_id'], "product_id": item['product_id']})
    return order_id

def check_regional_inventory(engine, region_code, quantity_threshold=50):
    query = text("""
        SELECT p.product_id, p.name, p.category, i.quantity, i.location_id
        FROM inventory i JOIN products p ON i.product_id = p.product_id
        WHERE i.region_code = :region_code AND i.quantity < :threshold;
    """)
    with engine.connect() as conn:
        return pd.read_sql(query, con=conn, params={"region_code": region_code, "threshold": quantity_threshold})

def get_sales_summary(engine):
    query = text("""
        SELECT DATE(o.order_time) AS sale_date, o.region_code, SUM(oi.qty * oi.unit_price) AS daily_revenue
        FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
        GROUP BY DATE(o.order_time), o.region_code ORDER BY sale_date DESC, daily_revenue DESC;
    """)
    with engine.connect() as conn:
        return pd.read_sql(query, con=conn)

def get_user_order_history(engine, user_id):
    query = text("""
        SELECT o.order_id, o.order_time, o.region_code, o.status, SUM(oi.qty * oi.unit_price) AS total_amount
        FROM orders AS o JOIN order_items AS oi ON o.order_id = oi.order_id
        WHERE o.user_id = :user_id AND o.order_time >= NOW() - INTERVAL 30 DAY
        GROUP BY o.order_id, o.order_time, o.region_code, o.status ORDER BY o.order_time DESC LIMIT 20;
    """)
    with engine.connect() as conn:
        return pd.read_sql(query, con=conn, params={"user_id": user_id})


# --- 2. Tkinter UI Class ---

class WarehouseApp:
    def __init__(self, root, engine):
        self.root = root
        self.engine = engine
        self.root.title("TiDB Global E-Commerce & Warehouse System")
        self.root.geometry("1100x800")
        
        style = ttk.Style()
        style.theme_use('clam')
        
        banner = tk.Label(root, text="Warehouse Management Dashboard", font=("Helvetica", 18, "bold"), bg="#1a365d", fg="white", pady=10)
        banner.pack(fill=tk.X)

        # --- VALIDATION REGISTRATION ---
        self.vcmd_int = (self.root.register(self.validate_int), '%P')
        self.vcmd_float = (self.root.register(self.validate_float), '%P')
        self.vcmd_email = (self.root.register(self.validate_email_chars), '%P')

        self.notebook = ttk.Notebook(root)
        self.notebook.pack(fill='both', expand=True, padx=10, pady=10)

        self.cart_items = []
        self.product_list = []
        self.products_df = pd.DataFrame()
        self.refresh_product_data()

        self.setup_place_order_tab()
        self.setup_add_inventory_tab() # Added missing inventory tab back in
        self.setup_add_user_tab()
        self.setup_add_product_tab()
        self.setup_inventory_tab()
        self.setup_sales_tab()
        self.setup_orders_tab()

    def refresh_product_data(self):
        try:
            self.products_df = get_all_products(self.engine)
            self.product_list = [f"{row['product_id']} - {row['name']}" for _, row in self.products_df.iterrows()]
            if hasattr(self, 'product_dropdown'):
                self.product_dropdown['values'] = self.product_list
            if hasattr(self, 'inv_product_dropdown'):
                self.inv_product_dropdown['values'] = self.product_list
        except Exception as e:
            print(f"Error fetching products: {e}")

    # --- INPUT VALIDATION RULES ---
    def validate_int(self, P):
        if P == "" or P.isdigit(): return True
        return False

    def validate_float(self, P):
        if P == "": return True
        try:
            float(P)
            return True
        except ValueError:
            return False

    def validate_email_chars(self, P):
        """Prevents spaces and multiple @ symbols from being typed."""
        if " " in P: return False
        if P.count('@') > 1: return False
        return True

    # --- 3. UI Tab Setups ---
    
    def setup_add_user_tab(self):
        tab = ttk.Frame(self.notebook)
        self.notebook.add(tab, text="Add New User")
        form_frame = ttk.LabelFrame(tab, text="User Details", padding=20)
        form_frame.pack(fill=tk.X, padx=20, pady=20)
        
        ttk.Label(form_frame, text="Full Name:").grid(row=0, column=0, sticky=tk.W, padx=5, pady=10)
        self.new_user_name = tk.StringVar()
        ttk.Entry(form_frame, textvariable=self.new_user_name, width=40).grid(row=0, column=1, padx=5, pady=10)
        
        ttk.Label(form_frame, text="Email Address:").grid(row=1, column=0, sticky=tk.W, padx=5, pady=10)
        self.new_user_email = tk.StringVar()
        ttk.Entry(form_frame, textvariable=self.new_user_email, width=40, validate="key", validatecommand=self.vcmd_email).grid(row=1, column=1, padx=5, pady=10)
        
        ttk.Label(form_frame, text="Region Code:").grid(row=2, column=0, sticky=tk.W, padx=5, pady=10)
        self.new_user_region = tk.StringVar(value="US-CA")
        ttk.Combobox(form_frame, textvariable=self.new_user_region, values=['US-CA', 'US-NY', 'EU-UK', 'AS-JP', 'AS-IN']).grid(row=2, column=1, sticky=tk.W, padx=5, pady=10)
        ttk.Button(form_frame, text="Create User", command=self.submit_new_user).grid(row=3, column=0, columnspan=2, pady=20)

    def setup_add_product_tab(self):
        tab = ttk.Frame(self.notebook)
        self.notebook.add(tab, text="Add New Product")
        form_frame = ttk.LabelFrame(tab, text="Product Details", padding=20)
        form_frame.pack(fill=tk.X, padx=20, pady=20)
        
        ttk.Label(form_frame, text="Product Name:").grid(row=0, column=0, sticky=tk.W, padx=5, pady=10)
        self.new_prod_name = tk.StringVar()
        ttk.Entry(form_frame, textvariable=self.new_prod_name, width=40).grid(row=0, column=1, padx=5, pady=10)
        
        ttk.Label(form_frame, text="Category:").grid(row=1, column=0, sticky=tk.W, padx=5, pady=10)
        self.new_prod_category = tk.StringVar(value="Electronics")
        ttk.Combobox(form_frame, textvariable=self.new_prod_category, values=['Electronics', 'Clothing', 'Home', 'Books', 'Toys']).grid(row=1, column=1, sticky=tk.W, padx=5, pady=10)
        
        ttk.Label(form_frame, text="Price ($):").grid(row=2, column=0, sticky=tk.W, padx=5, pady=10)
        self.new_prod_price = tk.StringVar(value="")
        ttk.Entry(form_frame, textvariable=self.new_prod_price, width=20, validate="key", validatecommand=self.vcmd_float).grid(row=2, column=1, sticky=tk.W, padx=5, pady=10)
        ttk.Button(form_frame, text="Create Product", command=self.submit_new_product).grid(row=3, column=0, columnspan=2, pady=20)

    def setup_add_inventory_tab(self):
        tab = ttk.Frame(self.notebook)
        self.notebook.add(tab, text="Add/Update Inventory")
        
        form_frame = ttk.LabelFrame(tab, text="Receive Inventory to Warehouse", padding=20)
        form_frame.pack(fill=tk.X, padx=20, pady=20)
        
        ttk.Label(form_frame, text="Select Product:").grid(row=0, column=0, sticky=tk.W, padx=5, pady=10)
        self.inv_prod_combo = tk.StringVar()
        self.inv_product_dropdown = ttk.Combobox(form_frame, textvariable=self.inv_prod_combo, state="readonly", width=40, values=self.product_list)
        self.inv_product_dropdown.grid(row=0, column=1, sticky=tk.W, padx=5, pady=10)
        
        ttk.Label(form_frame, text="Location ID:").grid(row=1, column=0, sticky=tk.W, padx=5, pady=10)
        self.inv_loc_id = tk.StringVar(value="1")
        ttk.Entry(form_frame, textvariable=self.inv_loc_id, width=15, validate="key", validatecommand=self.vcmd_int).grid(row=1, column=1, sticky=tk.W, padx=5, pady=10)
        
        ttk.Label(form_frame, text="Region Code:").grid(row=2, column=0, sticky=tk.W, padx=5, pady=10)
        self.inv_region = tk.StringVar(value="US-CA")
        ttk.Combobox(form_frame, textvariable=self.inv_region, values=['US-CA', 'US-NY', 'EU-UK', 'AS-JP', 'AS-IN']).grid(row=2, column=1, sticky=tk.W, padx=5, pady=10)

        ttk.Label(form_frame, text="Quantity to Add:").grid(row=3, column=0, sticky=tk.W, padx=5, pady=10)
        self.inv_qty = tk.StringVar(value="50")
        ttk.Entry(form_frame, textvariable=self.inv_qty, width=15, validate="key", validatecommand=self.vcmd_int).grid(row=3, column=1, sticky=tk.W, padx=5, pady=10)
        
        ttk.Button(form_frame, text="Submit Inventory to TiDB", command=self.submit_inventory).grid(row=4, column=0, columnspan=2, pady=20)

    def setup_place_order_tab(self):
        tab = ttk.Frame(self.notebook)
        self.notebook.add(tab, text="Place New Order")
        
        meta_frame = ttk.LabelFrame(tab, text="1. Order Information", padding=10)
        meta_frame.pack(fill=tk.X, padx=10, pady=5)
        ttk.Label(meta_frame, text="User ID:").grid(row=0, column=0, padx=5, pady=5)
        self.po_user_id = tk.StringVar(value="1")
        ttk.Entry(meta_frame, textvariable=self.po_user_id, width=15, validate="key", validatecommand=self.vcmd_int).grid(row=0, column=1, padx=5, pady=5)
        ttk.Label(meta_frame, text="Region Code:").grid(row=0, column=2, padx=5, pady=5)
        self.po_region = tk.StringVar(value="US-CA")
        ttk.Combobox(meta_frame, textvariable=self.po_region, values=['US-CA', 'US-NY', 'EU-UK', 'AS-JP', 'AS-IN']).grid(row=0, column=3, padx=5, pady=5)

        item_frame = ttk.LabelFrame(tab, text="2. Add Items to Order", padding=10)
        item_frame.pack(fill=tk.X, padx=10, pady=5)
        
        ttk.Label(item_frame, text="Select Product:").grid(row=0, column=0, padx=5, pady=5)
        self.po_prod_combo = tk.StringVar()
        self.product_dropdown = ttk.Combobox(item_frame, textvariable=self.po_prod_combo, values=self.product_list, width=35, state="readonly")
        self.product_dropdown.grid(row=0, column=1, padx=5, pady=5)
        self.product_dropdown.bind("<<ComboboxSelected>>", self.on_product_selected)

        ttk.Label(item_frame, text="Unit Price ($):").grid(row=0, column=2, padx=5, pady=5)
        self.po_price = tk.StringVar(value="0.00")
        ttk.Entry(item_frame, textvariable=self.po_price, width=10, state="readonly").grid(row=0, column=3, padx=5, pady=5)

        ttk.Label(item_frame, text="Quantity:").grid(row=0, column=4, padx=5, pady=5)
        self.po_qty = tk.StringVar(value="1")
        ttk.Entry(item_frame, textvariable=self.po_qty, width=10, validate="key", validatecommand=self.vcmd_int).grid(row=0, column=5, padx=5, pady=5)

        ttk.Label(item_frame, text="Loc ID:").grid(row=0, column=6, padx=5, pady=5)
        self.po_loc = tk.StringVar(value="1")
        ttk.Entry(item_frame, textvariable=self.po_loc, width=5, validate="key", validatecommand=self.vcmd_int).grid(row=0, column=7, padx=5, pady=5)

        ttk.Button(item_frame, text="Add to Cart", command=self.add_to_cart).grid(row=0, column=8, padx=15, pady=5)

        cart_frame = ttk.LabelFrame(tab, text="3. Order Summary", padding=10)
        cart_frame.pack(fill='both', expand=True, padx=10, pady=5)
        self.cart_tree = ttk.Treeview(cart_frame, columns=("Product ID", "Product Name", "Quantity", "Total Price", "Location"), show="headings")
        for col in ["Product ID", "Product Name", "Quantity", "Total Price", "Location"]:
            self.cart_tree.heading(col, text=col)
            self.cart_tree.column(col, anchor=tk.CENTER)
        self.cart_tree.pack(fill='both', expand=True, pady=5)

        action_frame = ttk.Frame(cart_frame)
        action_frame.pack(fill=tk.X, pady=5)
        ttk.Button(action_frame, text="Clear Cart", command=self.clear_cart).pack(side=tk.LEFT, padx=5)
        ttk.Button(action_frame, text="Submit Order to TiDB", command=self.submit_order).pack(side=tk.RIGHT, padx=5)

    def setup_inventory_tab(self):
        tab = ttk.Frame(self.notebook)
        self.notebook.add(tab, text="Regional Inventory Check")
        ctrl_frame = ttk.Frame(tab, padding=10)
        ctrl_frame.pack(fill=tk.X)
        ttk.Label(ctrl_frame, text="Region Code:").grid(row=0, column=0, padx=5, pady=5)
        self.region_var = tk.StringVar(value="US-CA")
        ttk.Combobox(ctrl_frame, textvariable=self.region_var, values=['US-CA', 'US-NY', 'EU-UK', 'AS-JP', 'AS-IN']).grid(row=0, column=1, padx=5, pady=5)
        ttk.Label(ctrl_frame, text="Threshold:").grid(row=0, column=2, padx=5, pady=5)
        self.threshold_var = tk.StringVar(value="50")
        ttk.Entry(ctrl_frame, textvariable=self.threshold_var, width=10, validate="key", validatecommand=self.vcmd_int).grid(row=0, column=3, padx=5, pady=5)
        ttk.Button(ctrl_frame, text="Check Inventory", command=self.load_inventory).grid(row=0, column=4, padx=15, pady=5)
        self.inv_tree = ttk.Treeview(tab)
        self.inv_tree.pack(fill='both', expand=True, padx=10, pady=10)

    def setup_sales_tab(self):
        tab = ttk.Frame(self.notebook)
        self.notebook.add(tab, text="Sales Summary")
        ttk.Button(tab, text="Refresh Sales Data", command=self.load_sales).pack(pady=10)
        self.sales_tree = ttk.Treeview(tab)
        self.sales_tree.pack(fill='both', expand=True, padx=10, pady=10)

    def setup_orders_tab(self):
        tab = ttk.Frame(self.notebook)
        self.notebook.add(tab, text="User Order History")
        ctrl_frame = ttk.Frame(tab, padding=10)
        ctrl_frame.pack(fill=tk.X)
        ttk.Label(ctrl_frame, text="User ID:").grid(row=0, column=0, padx=5, pady=5)
        self.history_user_id = tk.StringVar(value="10")
        ttk.Entry(ctrl_frame, textvariable=self.history_user_id, width=15, validate="key", validatecommand=self.vcmd_int).grid(row=0, column=1, padx=5, pady=5)
        ttk.Button(ctrl_frame, text="Search Orders", command=self.load_orders).grid(row=0, column=2, padx=15, pady=5)
        self.orders_tree = ttk.Treeview(tab)
        self.orders_tree.pack(fill='both', expand=True, padx=10, pady=10)

    # --- 4. Event Logic & Actions ---

    def submit_new_user(self):
        name = self.new_user_name.get().strip()
        email = self.new_user_email.get().strip()
        region = self.new_user_region.get()
        
        if not name or not email:
            messagebox.showwarning("Validation Error", "Please provide both Name and Email.")
            return
            
        if not re.match(r'^[\w\.-]+@[\w\.-]+\.\w+$', email):
            messagebox.showwarning("Validation Error", "Please provide a valid email address format.")
            return
            
        if email_exists(self.engine, email):
            messagebox.showwarning("Database Conflict", f"The email '{email}' is already registered.")
            return

        try:
            user_id = add_user(self.engine, name, email, region)
            messagebox.showinfo("Success", f"User '{name}' created successfully with User ID: {user_id}")
            self.new_user_name.set("")
            self.new_user_email.set("")
        except Exception as e:
            messagebox.showerror("Database Error", f"Failed to add user.\nError: {str(e)}")

    def submit_new_product(self):
        name = self.new_prod_name.get().strip()
        category = self.new_prod_category.get()
        price_str = self.new_prod_price.get()
        
        if not name or not price_str or float(price_str) <= 0:
            messagebox.showwarning("Validation Error", "Please provide a valid Product Name and a Price > $0.00.")
            return
            
        try:
            prod_id = add_product(self.engine, name, category, float(price_str))
            messagebox.showinfo("Success", f"Product '{name}' created successfully with Product ID: {prod_id}")
            self.refresh_product_data()
            self.new_prod_name.set("")
            self.new_prod_price.set("")
        except Exception as e:
            messagebox.showerror("Database Error", f"Failed to add product.\nError: {str(e)}")

    def submit_inventory(self):
        selected_str = self.inv_prod_combo.get()
        loc_str = self.inv_loc_id.get()
        qty_str = self.inv_qty.get()

        if not selected_str or not loc_str or not qty_str:
            messagebox.showwarning("Validation Error", "Please fill in all fields.")
            return

        prod_id = int(selected_str.split(" - ")[0])
        
        try:
            add_inventory(self.engine, prod_id, int(loc_str), self.inv_region.get(), int(qty_str))
            messagebox.showinfo("Success", f"Successfully added {qty_str} units of Product #{prod_id} to Location {loc_str}.")
            self.inv_prod_combo.set("")
            self.inv_qty.set("50")
        except Exception as e:
            messagebox.showerror("Database Error", f"Failed to update inventory.\nError: {str(e)}")

    def on_product_selected(self, event):
        selected_str = self.po_prod_combo.get()
        if not selected_str: return
        prod_id = int(selected_str.split(" - ")[0])
        product_row = self.products_df[self.products_df['product_id'] == prod_id]
        if not product_row.empty:
            self.po_price.set(f"{product_row.iloc[0]['price']:.2f}")

    def add_to_cart(self):
        selected_str = self.po_prod_combo.get()
        if not selected_str:
            messagebox.showwarning("Input Error", "Please select a product.")
            return
            
        qty_str = self.po_qty.get()
        loc_str = self.po_loc.get()

        if not qty_str or int(qty_str) <= 0:
            messagebox.showwarning("Validation Error", "Quantity must be greater than 0.")
            return
        if not loc_str:
            messagebox.showwarning("Validation Error", "Please provide a Location ID.")
            return
            
        qty = int(qty_str)
        loc_id = int(loc_str)
        prod_id = int(selected_str.split(" - ")[0])
        prod_name = selected_str.split(" - ", 1)[1]
        unit_price = float(self.po_price.get())
        
        is_available, msg = check_stock(self.engine, prod_id, loc_id, qty)
        if not is_available:
            messagebox.showerror("Inventory Constraint", msg)
            return

        total_price = qty * unit_price
        item = {'product_id': prod_id, 'qty': qty, 'unit_price': unit_price, 'location_id': loc_id}
        self.cart_items.append(item)
        
        self.cart_tree.insert("", "end", values=(prod_id, prod_name, qty, f"${total_price:.2f}", loc_id))
        self.product_dropdown.set('')
        self.po_price.set("0.00")
        self.po_qty.set("1")

    def clear_cart(self):
        self.cart_items = []
        self.cart_tree.delete(*self.cart_tree.get_children())

    def submit_order(self):
        if not self.cart_items:
            messagebox.showwarning("Empty Cart", "Please add items to the order first.")
            return

        uid_str = self.po_user_id.get()
        if not uid_str:
            messagebox.showwarning("Validation Error", "Please provide a User ID.")
            return
            
        uid = int(uid_str)
        if not user_exists(self.engine, uid):
            messagebox.showerror("Integrity Error", f"User ID {uid} does not exist in the database.")
            return

        try:
            order_id = place_order(self.engine, user_id=uid, region_code=self.po_region.get(), items=self.cart_items)
            messagebox.showinfo("Success", f"Order #{order_id} successfully placed in TiDB!\n\nInventory updated & 2PC Committed.")
            self.clear_cart()
        except Exception as e:
            messagebox.showerror("Transaction Failed", f"TiDB rollback initiated.\nError: {str(e)}")

    # --- 5. Read Functions ---
    def load_inventory(self):
        threshold_str = self.threshold_var.get()
        if not threshold_str:
            messagebox.showwarning("Input Error", "Please enter a valid threshold number.")
            return
        try:
            df = check_regional_inventory(self.engine, self.region_var.get(), int(threshold_str))
            self.populate_treeview(self.inv_tree, df)
        except Exception as e: messagebox.showerror("Database Error", str(e))

    def load_sales(self):
        try:
            df = get_sales_summary(self.engine)
            self.populate_treeview(self.sales_tree, df)
        except Exception as e: messagebox.showerror("Database Error", str(e))

    def load_orders(self):
        uid_str = self.history_user_id.get()
        if not uid_str:
            messagebox.showwarning("Input Error", "Please enter a User ID.")
            return
        try:
            df = get_user_order_history(self.engine, int(uid_str))
            self.populate_treeview(self.orders_tree, df)
        except Exception as e: 
            messagebox.showerror("Database Error", str(e))

    def populate_treeview(self, tree, df):
        tree.delete(*tree.get_children())
        tree["columns"] = list(df.columns)
        tree["show"] = "headings"
        for col in df.columns:
            tree.heading(col, text=str(col).replace("_", " ").title())
            tree.column(col, width=120, anchor=tk.CENTER)
        for index, row in df.iterrows():
            tree.insert("", "end", values=list(row))

if __name__ == "__main__":
    DATABASE_URI = 'mysql+pymysql://root:@127.0.0.1:4000/tidb_ecommerce'
    try:
        engine = create_engine(DATABASE_URI)
        root = tk.Tk()
        app = WarehouseApp(root, engine)
        root.mainloop()
    except Exception as e:
        print(f"Failed to connect to TiDB: {e}")

In [ ]:
#Run this part of code if you want to test the system vigourously and get data for workload 
# import time
# import random
# import threading
# import pandas as pd
# from sqlalchemy import create_engine, text

# # --- 1. Core Database Functions ---
# # (Reused from your application backend for standalone execution)

# def place_order(engine, user_id, region_code, items):
#     with engine.begin() as conn:
#         order_insert_stmt = text("INSERT INTO orders (user_id, region_code, status) VALUES (:user_id, :region_code, 'PENDING')")
#         result = conn.execute(order_insert_stmt, {"user_id": user_id, "region_code": region_code})
#         order_id = result.lastrowid

#         for item in items:
#             oi_stmt = text("INSERT INTO order_items (order_id, product_id, qty, unit_price) VALUES (:order_id, :product_id, :qty, :unit_price)")
#             conn.execute(oi_stmt, {"order_id": order_id, "product_id": item['product_id'], "qty": item['qty'], "unit_price": item['unit_price']})

#             inv_stmt = text("UPDATE inventory SET quantity = quantity - :qty WHERE location_id = :location_id AND product_id = :product_id")
#             conn.execute(inv_stmt, {"qty": item['qty'], "location_id": item['location_id'], "product_id": item['product_id']})
#     return order_id

# def check_regional_inventory(engine, region_code, quantity_threshold=50):
#     query = text("""
#         SELECT p.product_id, p.name, p.category, i.quantity, i.location_id
#         FROM inventory i JOIN products p ON i.product_id = p.product_id
#         WHERE i.region_code = :region_code AND i.quantity < :threshold;
#     """)
#     with engine.connect() as conn:
#         return pd.read_sql(query, con=conn, params={"region_code": region_code, "threshold": quantity_threshold})

# def get_sales_summary(engine):
#     query = text("""
#         SELECT DATE(o.order_time) AS sale_date, o.region_code, SUM(oi.qty * oi.unit_price) AS daily_revenue
#         FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
#         GROUP BY DATE(o.order_time), o.region_code ORDER BY sale_date DESC, daily_revenue DESC;
#     """)
#     with engine.connect() as conn:
#         return pd.read_sql(query, con=conn)

# def get_user_order_history(engine, user_id):
#     query = text("""
#         SELECT o.order_id, o.order_time, o.region_code, o.status, SUM(oi.qty * oi.unit_price) AS total_amount
#         FROM orders AS o JOIN order_items AS oi ON o.order_id = oi.order_id
#         WHERE o.user_id = :user_id AND o.order_time >= NOW() - INTERVAL 30 DAY
#         GROUP BY o.order_id, o.order_time, o.region_code, o.status ORDER BY o.order_time DESC LIMIT 20;
#     """)
#     with engine.connect() as conn:
#         return pd.read_sql(query, con=conn, params={"user_id": user_id})


# # --- 2. Load Generator Class ---

# class TiDBLoadGenerator:
#     def __init__(self, uri):
#         self.engine = create_engine(uri)
#         self.regions = ['US-CA', 'US-NY', 'EU-UK', 'AS-JP', 'AS-IN']
#         self.locations = {'US-CA': 1, 'US-NY': 2, 'EU-UK': 3, 'AS-JP': 4, 'AS-IN': 5}
        
#         print("Fetching database metadata to ensure valid traffic generation...")
#         self.load_metadata()

#     def load_metadata(self):
#         """Fetches valid users and products so we don't trigger integrity errors."""
#         with self.engine.connect() as conn:
#             # Fetch valid user IDs
#             users_df = pd.read_sql("SELECT user_id FROM users", con=conn)
#             self.valid_user_ids = users_df['user_id'].tolist()
            
#             # Fetch products with their actual prices
#             self.products_df = pd.read_sql("SELECT product_id, price FROM products", con=conn)

#     def simulate_write_traffic(self):
#         """Generates continuous INSERT/UPDATE operations to test 2PC and Raft writes."""
#         print("Write thread started.")
#         while True:
#             try:
#                 user_id = random.choice(self.valid_user_ids)
#                 region = random.choice(self.regions)
#                 loc_id = self.locations[region]
                
#                 # Pick 1 to 5 random unique products from our dataframe
#                 sample_products = self.products_df.sample(n=random.randint(1, 5))
                
#                 items = []
#                 for _, row in sample_products.iterrows():
#                     items.append({
#                         'product_id': int(row['product_id']),
#                         'qty': random.randint(1, 3),
#                         'unit_price': float(row['price']),
#                         'location_id': loc_id
#                     })
                    
#                 place_order(self.engine, user_id, region, items)
#                 print(f"[WRITE] Order placed successfully in {region} for User {user_id}")
                
#             except Exception as e:
#                 # If inventory goes negative or deadlocks occur, catch and print it
#                 print(f"[WRITE FAILED] {str(e)[:100]}...")
                
#             # Sleep briefly to throttle load (adjust this depending on your PC's power)
#             time.sleep(random.uniform(0.1, 0.5)) 

#     def simulate_read_traffic(self):
#         """Generates heavy READ traffic testing point lookups and aggregations."""
#         print("Read thread started.")
#         while True:
#             try:
#                 action = random.choice(['history', 'inventory', 'sales'])
                
#                 if action == 'history':
#                     user_id = random.choice(self.valid_user_ids)
#                     get_user_order_history(self.engine, user_id)
#                     print(f"[READ] Fetched order history for User {user_id}")
                    
#                 elif action == 'inventory':
#                     region = random.choice(self.regions)
#                     check_regional_inventory(self.engine, region, threshold=random.randint(10, 50))
#                     print(f"[READ] Scanned regional inventory for {region}")
                    
#                 elif action == 'sales':
#                     get_sales_summary(self.engine)
#                     print("[READ] Generated global sales aggregation summary")
                    
#             except Exception as e:
#                 print(f"[READ FAILED] {str(e)[:100]}...")
                
#             # Aggregations are heavier on the CPU/Coprocessor, run less frequently
#             time.sleep(random.uniform(0.5, 2.0))

#     def run_experiment(self, duration_seconds):
#         print(f"--- Starting TiDB Simulated Traffic for {duration_seconds} seconds ---")
        
#         # Start Threads as Daemons so they die when the main thread finishes
#         write_thread = threading.Thread(target=self.simulate_write_traffic, daemon=True)
#         read_thread = threading.Thread(target=self.simulate_read_traffic, daemon=True)
        
#         write_thread.start()
#         read_thread.start()
        
#         # Keep main thread alive for the duration of the experiment
#         time.sleep(duration_seconds)
#         print("--- Experiment Completed ---")
#         print("Check Grafana for Region Hotness, Raft Proposes, and end-to-end Latency metrics!")


# if __name__ == "__main__":
#     # Ensure this URI matches the one in your Jupyter notebook and Tkinter app
#     DATABASE_URI = 'mysql+pymysql://root:@127.0.0.1:4000/tidb_ecommerce'
    
#     try:
#         generator = TiDBLoadGenerator(DATABASE_URI)
        
#         # Run the experiment for 5 minutes (300 seconds)
#         # You can increase this to gather more sustained metrics for Grafana
#         generator.run_experiment(duration_seconds=300)
        
#     except Exception as e:
#         print(f"Failed to initialize load generator: {e}")